# Smart Museum Guide - AI service (Google Colab)

Translates exhibit copy and narrates it for the museum CMS.

```text
Admin saves an exhibit
  -> backend queues one task per language
  -> POST /v1/jobs to this notebook (via a Cloudflare tunnel)
  -> queue: translate (TranslateGemma) -> narrate (VoxCPM2, OmniVoice for cs/uk)
  -> signed webhook -> backend stores the translation and the MP3 for that exhibit
```

### Before running
1. **Runtime -> Change runtime type -> GPU.** L4 or A100 is recommended. A T4 works: the translator is loaded in 4-bit automatically.
2. **Colab secrets** (key icon in the left bar), with notebook access switched on:
   - `AI_SERVICE_SECRET`: the same value as `AI_SERVICE_SECRET` in `backend/api/.env`.
   - `HF_TOKEN`: a Hugging Face token. First accept the Gemma license on https://huggingface.co/google/translategemma-4b-it.
3. **Make the backend API reachable from Colab.** On the machine running the API: `cloudflared tunnel --url http://localhost:3001`. Put `https://<random>.trycloudflare.com/api` in the form below.
4. **Runtime -> Run all**, then keep this tab open. Colab stops idle or long sessions; after a restart just run all again. Tasks that were waiting are picked up automatically.

In [ ]:
!nvidia-smi || echo 'No GPU: switch the runtime type, or tick MOCK_MODELS to test the pipeline.'

In [ ]:
#@title Configuration
BACKEND_API_URL = ""  #@param {type:"string"}
TRANSLATION_MODEL = "google/translategemma-4b-it"  #@param ["google/translategemma-4b-it", "google/translategemma-12b-it", "xiaomi-research/MiLMMT-46-4B-v1.0"]
TTS_MODEL = "openbmb/VoxCPM2"  #@param {type:"string"}
FALLBACK_TTS_MODEL = "k2-fsa/OmniVoice"  #@param ["k2-fsa/OmniVoice", ""]
VOICE_DESCRIPTION = "A warm, clear middle-aged female museum guide, calm and friendly storytelling tone"  #@param {type:"string"}
NARRATION_STYLE = ""  #@param {type:"string"}
#@markdown Test the pipeline without GPU models (a tone instead of speech, "[en] ..." instead of a translation):
MOCK_MODELS = False  #@param {type:"boolean"}

import os

def colab_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name) or ""
    except Exception:
        return os.environ.get(name, "")

os.environ.update({
    "BACKEND_API_URL": BACKEND_API_URL.strip().rstrip("/"),
    "AI_SERVICE_SECRET": colab_secret("AI_SERVICE_SECRET"),
    "TRANSLATION_MODEL": TRANSLATION_MODEL,
    "TTS_MODEL": TTS_MODEL,
    "FALLBACK_TTS_MODEL": FALLBACK_TTS_MODEL,
    "VOICE_DESCRIPTION": VOICE_DESCRIPTION,
    "NARRATION_STYLE": NARRATION_STYLE,
    "MOCK_MODELS": "1" if MOCK_MODELS else "0",
    "WORK_DIR": "/content/museum-ai",
})
if colab_secret("HF_TOKEN"):
    os.environ["HF_TOKEN"] = colab_secret("HF_TOKEN")

assert os.environ["BACKEND_API_URL"].endswith("/api"), "BACKEND_API_URL must look like https://....trycloudflare.com/api"
assert len(os.environ["AI_SERVICE_SECRET"]) >= 16, "Add the AI_SERVICE_SECRET Colab secret (16+ characters)"
print("Backend API:", os.environ["BACKEND_API_URL"])

In [ ]:
#@title Install dependencies (a few minutes the first time)
import subprocess, sys

packages = ["fastapi>=0.115", "uvicorn>=0.30", "httpx>=0.27", "pydantic>=2.7", "soundfile>=0.12"]
if not MOCK_MODELS:
    packages += ["transformers>=4.57", "accelerate>=1.0", "bitsandbytes>=0.45", "voxcpm>=2.0"]
    if FALLBACK_TTS_MODEL:
        packages += ["omnivoice"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)
print("Installed:", ", ".join(packages))

## Service code

Generated from `ai-services/museum_ai` by `scripts/build_notebook.py` - edit the package, not these cells.

In [ ]:
import os
os.makedirs('museum_ai', exist_ok=True)

In [ ]:
%%writefile museum_ai/__init__.py
"""Smart Museum Guide AI service: translate exhibit copy and narrate it.

The backend API hands jobs to this service; each task is translated from the
primary copy, narrated, and reported back through a signed webhook.
"""

__version__ = "1.0.0"

In [ ]:
%%writefile museum_ai/config.py
"""Runtime settings, read from environment variables (set them in the notebook)."""

from __future__ import annotations

import os
from dataclasses import dataclass
from typing import Optional, Tuple


def _text(name: str, default: Optional[str] = None) -> Optional[str]:
    value = os.environ.get(name)
    if value is None or not value.strip():
        return default
    return value.strip()


def _flag(name: str, default: bool) -> bool:
    value = _text(name)
    if value is None:
        return default
    return value.lower() in {"1", "true", "yes", "on"}


def _codes(name: str, default: Tuple[str, ...]) -> Tuple[str, ...]:
    value = os.environ.get(name)
    if value is None:
        return default
    return tuple(code.strip().lower() for code in value.split(",") if code.strip())


def _number(name: str, default: float) -> float:
    value = _text(name)
    return float(value) if value is not None else default


@dataclass
class Settings:
    #: Backend API base URL including `/api`, reachable from Colab
    #: (e.g. `https://abc.trycloudflare.com/api`).
    backend_api_url: str
    #: Must equal AI_SERVICE_SECRET in backend/api/.env.
    secret: str

    host: str = "127.0.0.1"
    port: int = 8000
    #: Public URL of this service. Unset: open a Cloudflare quick tunnel.
    public_url: Optional[str] = None
    heartbeat_seconds: float = 20.0

    translation_model: str = "google/translategemma-4b-it"
    #: `auto` quantises the translator to 4-bit on GPUs under 20 GB (e.g. a T4).
    translation_quantization: str = "auto"
    tts_model: str = "openbmb/VoxCPM2"
    #: Used for languages the primary TTS model does not cover. Empty disables it.
    fallback_tts_model: Optional[str] = "k2-fsa/OmniVoice"
    #: Languages narrated by the fallback model, and therefore offered to the CMS.
    fallback_tts_languages: Tuple[str, ...] = ("cs", "uk")
    #: VoxCPM2 voice design prompt; one reference voice per language is created from it.
    voice_description: str = (
        "A warm, clear middle-aged female museum guide, calm and friendly storytelling tone"
    )
    #: Optional VoxCPM2 style instruction applied to every narrated sentence.
    narration_style: Optional[str] = None
    #: OmniVoice speaker attributes (gender, age, pitch).
    omnivoice_instruct: str = "female, middle-aged, moderate pitch"
    #: Your own reference recording (3-10 s) cloned for every language instead.
    reference_voice_path: Optional[str] = None
    reference_voice_text: Optional[str] = None
    #: torch.compile for VoxCPM2: faster once warm, but slow to start and fragile on older GPUs.
    voxcpm_optimize: bool = False

    max_chunk_chars: int = 220
    mp3_bitrate: str = "64k"
    work_dir: str = ".museum-ai"
    #: Replace every model with a fast fake, to test the pipeline without a GPU.
    mock_models: bool = False

    @classmethod
    def from_env(cls) -> "Settings":
        backend_api_url = _text("BACKEND_API_URL")
        secret = _text("AI_SERVICE_SECRET")
        if not backend_api_url:
            raise ValueError("BACKEND_API_URL is required (e.g. https://abc.trycloudflare.com/api).")
        if not secret or len(secret) < 16:
            raise ValueError("AI_SERVICE_SECRET is required and must be at least 16 characters.")

        defaults = cls(backend_api_url="", secret="")
        # Unset keeps the default; an empty value disables the fallback model.
        fallback = os.environ.get("FALLBACK_TTS_MODEL")
        return cls(
            backend_api_url=backend_api_url.rstrip("/"),
            secret=secret,
            host=_text("HOST", defaults.host),
            port=int(_number("PORT", defaults.port)),
            public_url=(_text("PUBLIC_URL") or "").rstrip("/") or None,
            heartbeat_seconds=_number("HEARTBEAT_SECONDS", defaults.heartbeat_seconds),
            translation_model=_text("TRANSLATION_MODEL", defaults.translation_model),
            translation_quantization=_text(
                "TRANSLATION_QUANTIZATION", defaults.translation_quantization
            ).lower(),
            tts_model=_text("TTS_MODEL", defaults.tts_model),
            fallback_tts_model=defaults.fallback_tts_model
            if fallback is None
            else (fallback.strip() or None),
            fallback_tts_languages=_codes("FALLBACK_TTS_LANGUAGES", defaults.fallback_tts_languages),
            voice_description=_text("VOICE_DESCRIPTION", defaults.voice_description),
            narration_style=_text("NARRATION_STYLE"),
            omnivoice_instruct=_text("OMNIVOICE_INSTRUCT", defaults.omnivoice_instruct),
            reference_voice_path=_text("REFERENCE_VOICE_PATH"),
            reference_voice_text=_text("REFERENCE_VOICE_TEXT"),
            voxcpm_optimize=_flag("VOXCPM_OPTIMIZE", defaults.voxcpm_optimize),
            max_chunk_chars=int(_number("MAX_CHUNK_CHARS", defaults.max_chunk_chars)),
            mp3_bitrate=_text("MP3_BITRATE", defaults.mp3_bitrate),
            work_dir=_text("WORK_DIR", defaults.work_dir),
            mock_models=_flag("MOCK_MODELS", defaults.mock_models),
        )

In [ ]:
%%writefile museum_ai/signing.py
"""Request signatures shared with the backend (backend/api/src/localization/ai-service-signature.ts).

Header: `X-Museum-Signature: t=<unix seconds>,v1=<hex HMAC-SHA256 of "<t>.<raw body>">`.
"""

from __future__ import annotations

import hashlib
import hmac
import time
from typing import Optional

SIGNATURE_HEADER = "X-Museum-Signature"
TOLERANCE_SECONDS = 300


def sign(secret: str, body: bytes, now: Optional[float] = None) -> str:
    timestamp = int(time.time() if now is None else now)
    return f"t={timestamp},v1={_digest(secret, timestamp, body)}"


def verify(
    secret: str,
    header: Optional[str],
    body: bytes,
    now: Optional[float] = None,
    tolerance: int = TOLERANCE_SECONDS,
) -> bool:
    if not header:
        return False
    parts = dict(part.strip().split("=", 1) for part in header.split(",") if "=" in part)
    try:
        timestamp = int(parts.get("t", ""))
    except ValueError:
        return False
    signature = parts.get("v1", "")
    if len(signature) != 64:
        return False
    current = time.time() if now is None else now
    if abs(current - timestamp) > tolerance:
        return False
    return hmac.compare_digest(_digest(secret, timestamp, body), signature)


def _digest(secret: str, timestamp: int, body: bytes) -> str:
    return hmac.new(secret.encode(), f"{timestamp}.".encode() + body, hashlib.sha256).hexdigest()

In [ ]:
%%writefile museum_ai/schemas.py
"""Job payload sent by the backend (see AiJobRequest in backend/api/src/localization/ai-service.client.ts)."""

from __future__ import annotations

from typing import List, Optional

from pydantic import BaseModel, Field


class SourceCopy(BaseModel):
    languageCode: str
    title: str
    shortDescription: Optional[str] = None
    description: Optional[str] = None


class TaskSpec(BaseModel):
    taskId: str
    languageCode: str
    #: False for the primary language: narrate the source copy as it is.
    translate: bool


class JobRequest(BaseModel):
    exhibitId: str
    exhibitCode: str
    source: SourceCopy
    tasks: List[TaskSpec] = Field(min_length=1)

In [ ]:
%%writefile museum_ai/languages.py
"""Languages this service can offer, using the CMS language codes.

A language is offered only when the loaded translation model AND a narration
model both cover it; the result is reported to the backend with every heartbeat
(see `supported_languages`), and the CMS and visitor apps list exactly those.
Display names (native spellings) live in backend/api/src/languages/language-catalog.ts.
"""

from typing import Callable, Iterable, List

#: English names, as the translation prompts expect them. Order is not significant.
LANGUAGE_NAMES = {
    "vi": "Vietnamese",
    "en": "English",
    "ja": "Japanese",
    "ko": "Korean",
    "zh": "Chinese (Simplified)",
    "zh-hant": "Chinese (Traditional)",
    "th": "Thai",
    "id": "Indonesian",
    "ms": "Malay",
    "km": "Khmer",
    "lo": "Lao",
    "my": "Burmese",
    "fil": "Filipino",
    "hi": "Hindi",
    "ar": "Arabic",
    "he": "Hebrew",
    "tr": "Turkish",
    "fr": "French",
    "de": "German",
    "es": "Spanish",
    "it": "Italian",
    "pt": "Portuguese",
    "nl": "Dutch",
    "ru": "Russian",
    "uk": "Ukrainian",
    "pl": "Polish",
    "cs": "Czech",
    "el": "Greek",
    "sv": "Swedish",
    "da": "Danish",
    "no": "Norwegian",
    "fi": "Finnish",
    "sw": "Swahili",
}

#: Spoken once per language to create that language's reference voice.
VOICE_SAMPLE_SENTENCE = "Welcome to the museum. Let me tell you the story behind this remarkable exhibit."


def language_name(code: str) -> str:
    return LANGUAGE_NAMES.get(code, code)


def supported_languages(translatable: Iterable[str], narratable: Callable[[str], bool]) -> List[str]:
    """Codes both models cover, e.g. `supported_languages(translator.languages, narrator.supports)`."""
    translatable = set(translatable)
    return [code for code in LANGUAGE_NAMES if code in translatable and narratable(code)]

In [ ]:
%%writefile museum_ai/text.py
"""Text helpers: narration script and sentence-aware chunking.

TTS models stay stable on short inputs, so narration is synthesised sentence by
sentence (grouped up to `max_chars`) and the pieces are joined afterwards.
"""

from __future__ import annotations

import re
from typing import List, Optional

# Terminators followed by a space (Latin, Cyrillic, Arabic, Devanagari), and
# terminators of scripts that do not put a space after them (CJK, Khmer).
_SPACED_TERMINATORS = ".!?;…؟۔।॥"
_UNSPACED_TERMINATORS = "。！？；។៕"
_TERMINATORS = _SPACED_TERMINATORS + _UNSPACED_TERMINATORS
_SENTENCE_BOUNDARY = re.compile(
    rf"(?:(?<=[{re.escape(_SPACED_TERMINATORS)}])|(?<=[{re.escape(_SPACED_TERMINATORS)}][\"'”’)\]]))\s+"
    rf"|(?<=[{_UNSPACED_TERMINATORS}])\s*"
)


def narration_text(title: str, short_description: Optional[str], description: Optional[str]) -> str:
    """Title, summary and body as one script, each part ending like a sentence."""
    parts = [part.strip() for part in (title, short_description, description) if part and part.strip()]
    script = []
    for part in parts:
        script.append(part if part[-1] in _TERMINATORS else f"{part}.")
    return "\n".join(script)


def split_paragraphs(text: str) -> List[str]:
    return [paragraph.strip() for paragraph in re.split(r"\n\s*\n", text) if paragraph.strip()]


def split_sentences(text: str) -> List[str]:
    sentences: List[str] = []
    for line in text.splitlines():
        sentences.extend(piece.strip() for piece in _SENTENCE_BOUNDARY.split(line) if piece.strip())
    return sentences


def chunk_text(text: str, max_chars: int) -> List[str]:
    """Groups whole sentences into chunks of at most `max_chars` characters.

    A sentence longer than the limit is split on spaces, or hard-split for
    scripts written without spaces.
    """
    chunks: List[str] = []
    current = ""
    for sentence in split_sentences(text):
        for piece in _split_long(sentence, max_chars):
            candidate = f"{current} {piece}".strip() if current else piece
            if len(candidate) <= max_chars:
                current = candidate
            else:
                chunks.append(current)
                current = piece
    if current:
        chunks.append(current)
    return chunks


def _split_long(sentence: str, max_chars: int) -> List[str]:
    pieces: List[str] = []
    rest = sentence
    while len(rest) > max_chars:
        cut = rest.rfind(" ", 0, max_chars + 1)
        if cut <= 0:
            cut = max_chars
        pieces.append(rest[:cut].strip())
        rest = rest[cut:].strip()
    if rest:
        pieces.append(rest)
    return pieces

In [ ]:
%%writefile museum_ai/translation.py
"""Machine translation of exhibit copy.

Default model: google/translategemma-4b-it (Gemma license, gated - accept it on
Hugging Face and set HF_TOKEN). Its training data covers every language in
LANGUAGE_NAMES (TranslateGemma technical report, appendix C: Khmer, Lao, Malay,
Burmese, Filipino, Czech, Ukrainian... are trained from English). The 12b/27b
variants trade speed for quality.

Alternative: xiaomi-research/MiLMMT-46-4B-v1.0 (Gemma license, not gated),
reported stronger than TranslateGemma but without Ukrainian and Swahili.
"""

from __future__ import annotations

import logging
from typing import Dict, FrozenSet, List, Optional, Protocol

from .languages import LANGUAGE_NAMES, language_name
from .schemas import SourceCopy
from .text import split_paragraphs, split_sentences

logger = logging.getLogger(__name__)

#: Both models accept roughly 2K tokens; paragraphs longer than this are split by sentence.
MAX_SEGMENT_CHARS = 1500

#: Limits of the backend webhook DTO (TranslatedCopyDto).
TITLE_MAX = 200
SHORT_DESCRIPTION_MAX = 500


class Translator(Protocol):
    model_id: str
    #: CMS language codes the model can translate between.
    languages: FrozenSet[str]

    def translate(self, text: str, source: str, target: str) -> str: ...


def translate_copy(translator: Translator, source: SourceCopy, target: str) -> Dict[str, Optional[str]]:
    """Translates title, summary and body; the body paragraph by paragraph."""
    src = source.languageCode

    def translate_block(text: Optional[str]) -> Optional[str]:
        if not text or not text.strip():
            return None
        paragraphs = [
            " ".join(translator.translate(segment, src, target) for segment in _segments(paragraph))
            for paragraph in split_paragraphs(text)
        ]
        return "\n\n".join(paragraphs) or None

    title = translate_block(source.title) or source.title
    short_description = translate_block(source.shortDescription)
    return {
        "title": _clip(title, TITLE_MAX),
        "shortDescription": _clip(short_description, SHORT_DESCRIPTION_MAX),
        "description": translate_block(source.description),
    }


def _segments(paragraph: str) -> List[str]:
    if len(paragraph) <= MAX_SEGMENT_CHARS:
        return [paragraph]
    segments: List[str] = []
    current = ""
    for sentence in split_sentences(paragraph):
        if current and len(current) + len(sentence) + 1 > MAX_SEGMENT_CHARS:
            segments.append(current)
            current = sentence
        else:
            current = f"{current} {sentence}".strip()
    if current:
        segments.append(current)
    return segments


def _clip(text: Optional[str], limit: int) -> Optional[str]:
    if text is None or len(text) <= limit:
        return text
    cut = text.rfind(" ", 0, limit - 1)
    return text[: cut if cut > limit // 2 else limit - 1].rstrip() + "…"


class MockTranslator:
    """Prefixes the target code; used with MOCK_MODELS=1."""

    model_id = "mock-translator"
    languages: FrozenSet[str] = frozenset(LANGUAGE_NAMES)

    def translate(self, text: str, source: str, target: str) -> str:
        return f"[{target}] {text}"


class TranslateGemmaTranslator:
    #: CMS codes that TranslateGemma's chat template knows under another name.
    CODES = {"zh": "zh-CN", "zh-hant": "zh-TW", "fil": "fil-PH"}
    languages: FrozenSet[str] = frozenset(LANGUAGE_NAMES)

    def __init__(self, model_id: str, quantization: str = "auto") -> None:
        import torch
        from transformers import AutoModelForImageTextToText, AutoProcessor

        self.model_id = model_id
        self._torch = torch
        self.processor = AutoProcessor.from_pretrained(model_id)
        self.dtype = _float_dtype(torch)
        self.model = AutoModelForImageTextToText.from_pretrained(
            model_id, device_map="auto", **_load_options(torch, quantization, self.dtype)
        )
        self.model.eval()

    def translate(self, text: str, source: str, target: str) -> str:
        messages = [
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "source_lang_code": self.CODES.get(source, source),
                        "target_lang_code": self.CODES.get(target, target),
                        "text": text,
                    }
                ],
            }
        ]
        inputs = self.processor.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt",
        ).to(self.model.device, dtype=self.dtype)
        input_length = inputs["input_ids"].shape[-1]
        with self._torch.inference_mode():
            output = self.model.generate(
                **inputs, do_sample=False, max_new_tokens=_max_new_tokens(input_length)
            )
        return self.processor.decode(output[0][input_length:], skip_special_tokens=True).strip()


class MiLMMTTranslator:
    #: Names from the model card; the prompt must use them verbatim.
    NAMES = {
        "zh": "Chinese (Simplified)",
        "zh-hant": "Chinese (Traditional)",
        "fil": "Tagalog",
    }
    #: The 46 model-card languages, minus the CMS languages it lacks.
    languages: FrozenSet[str] = frozenset(LANGUAGE_NAMES) - {"uk", "sw"}

    def __init__(self, model_id: str, quantization: str = "auto") -> None:
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer

        self.model_id = model_id
        self._torch = torch
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        dtype = _float_dtype(torch)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id, device_map="auto", **_load_options(torch, quantization, dtype)
        )
        self.model.eval()

    def translate(self, text: str, source: str, target: str) -> str:
        for code in (source, target):
            if code not in self.languages:
                raise ValueError(f"{self.model_id} does not support {language_name(code)}.")
        src = self.NAMES.get(source, language_name(source))
        tgt = self.NAMES.get(target, language_name(target))
        prompt = f"Translate this from {src} to {tgt}:\n{src}: {text}\n{tgt}:"
        inputs = self.tokenizer(prompt, add_special_tokens=False, return_tensors="pt").to(self.model.device)
        input_length = inputs["input_ids"].shape[-1]
        with self._torch.inference_mode():
            output = self.model.generate(
                **inputs, do_sample=False, max_new_tokens=_max_new_tokens(input_length)
            )
        decoded = self.tokenizer.decode(output[0][input_length:], skip_special_tokens=True)
        # Stop at a hallucinated follow-up prompt, if any.
        return decoded.split("\nTranslate this from", 1)[0].strip()


def load_translator(model_id: str, quantization: str, mock: bool) -> Translator:
    if mock:
        return MockTranslator()
    logger.info("Loading translation model %s", model_id)
    if "milmmt" in model_id.lower():
        return MiLMMTTranslator(model_id, quantization)
    return TranslateGemmaTranslator(model_id, quantization)


def _float_dtype(torch):
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
        return torch.bfloat16
    return torch.float16 if torch.cuda.is_available() else torch.float32


def _load_options(torch, quantization: str, dtype) -> dict:
    quantize = quantization == "4bit"
    if quantization == "auto" and torch.cuda.is_available():
        total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
        quantize = total_gb < 20
    if not quantize:
        return {"dtype": dtype}

    from transformers import BitsAndBytesConfig

    logger.info("Quantising the translation model to 4-bit to leave GPU memory for TTS")
    return {
        "quantization_config": BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=dtype,
        )
    }


def _max_new_tokens(input_length: int) -> int:
    # Translations rarely need more than ~2x the prompt, but CJK <-> Latin can expand.
    return min(2048, max(128, input_length * 3))

In [ ]:
%%writefile museum_ai/tts.py
"""Narration: text-to-speech with a consistent voice per language.

Primary model: openbmb/VoxCPM2 (Apache-2.0, 30 languages incl. vi, th, km, lo,
id, ms, fil). The languages listed in FALLBACK_TTS_LANGUAGES (cs, uk by default)
use k2-fsa/OmniVoice instead (600+ languages, but only these are offered, as
quality varies widely; its weights are CC-BY-NC - fine for research, not for
commercial use).

Voice consistency: on first use of a language, the engine designs a reference
voice from the voice description by speaking a short sample sentence in that
language. Every narration in that language then clones this reference, so all
exhibits share one guide voice, and cloning within the same language avoids a
foreign accent.
"""

from __future__ import annotations

import json
import logging
import math
import threading
from dataclasses import dataclass
from pathlib import Path
from typing import Callable, Dict, FrozenSet, Iterable, List, Optional, Protocol, Tuple

import numpy as np

from .languages import VOICE_SAMPLE_SENTENCE
from .text import chunk_text
from .translation import Translator

logger = logging.getLogger(__name__)

PAUSE_BETWEEN_CHUNKS_SECONDS = 0.25


@dataclass
class Voice:
    wav_path: str
    transcript: str


class SpeechEngine(Protocol):
    model_id: str
    sample_rate: int

    def supports(self, language: str) -> bool: ...

    def design(self, text: str, language: str) -> np.ndarray: ...

    def clone(self, text: str, language: str, voice: Voice) -> np.ndarray: ...


class VoxCPM2Engine:
    #: From the VoxCPM2 model card (fil = Tagalog, zh-hant read as Chinese).
    LANGUAGES = {
        "ar", "my", "zh", "zh-hant", "da", "nl", "en", "fi", "fr", "de", "el", "he", "hi", "id",
        "it", "ja", "km", "ko", "lo", "ms", "no", "pl", "pt", "ru", "es", "sw", "sv", "fil",
        "th", "tr", "vi",
    }  # fmt: skip

    def __init__(self, model_id: str, voice_description: str, style: Optional[str], optimize: bool):
        from voxcpm import VoxCPM

        self.model_id = model_id
        self.voice_description = voice_description
        self.style = style
        self.model = VoxCPM.from_pretrained(model_id, load_denoiser=False, optimize=optimize)
        self.sample_rate = int(self.model.tts_model.sample_rate)

    def supports(self, language: str) -> bool:
        return language in self.LANGUAGES

    def design(self, text: str, language: str) -> np.ndarray:
        # Voice design: the description goes in parentheses before the text.
        return self._generate(f"({self.voice_description}){text}")

    def clone(self, text: str, language: str, voice: Voice) -> np.ndarray:
        prefix = f"({self.style})" if self.style else ""
        return self._generate(f"{prefix}{text}", reference_wav_path=voice.wav_path)

    def _generate(self, text: str, **kwargs) -> np.ndarray:
        wav = self.model.generate(text=text, cfg_value=2.0, inference_timesteps=10, seed=42, **kwargs)
        return np.asarray(wav, dtype=np.float32)


class OmniVoiceEngine:
    sample_rate = 24000
    CODES = {"zh-hant": "zh"}

    def __init__(self, model_id: str, instruct: str):
        import torch
        from omnivoice import OmniVoice

        self.model_id = model_id
        self.instruct = instruct
        self.model = OmniVoice.from_pretrained(
            model_id,
            device_map="cuda:0" if torch.cuda.is_available() else "cpu",
            dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        )

    def supports(self, language: str) -> bool:
        return True

    def design(self, text: str, language: str) -> np.ndarray:
        audio = self.model.generate(text=text, language=self._code(language), instruct=self.instruct)
        return np.asarray(audio[0], dtype=np.float32)

    def clone(self, text: str, language: str, voice: Voice) -> np.ndarray:
        audio = self.model.generate(
            text=text,
            language=self._code(language),
            ref_audio=voice.wav_path,
            ref_text=voice.transcript,
        )
        return np.asarray(audio[0], dtype=np.float32)

    def _code(self, language: str) -> str:
        return self.CODES.get(language, language)


class MockEngine:
    """A soft tone whose length follows the text; used with MOCK_MODELS=1."""

    model_id = "mock-tts"
    sample_rate = 24000

    def supports(self, language: str) -> bool:
        return True

    def design(self, text: str, language: str) -> np.ndarray:
        return self._tone(text)

    def clone(self, text: str, language: str, voice: Voice) -> np.ndarray:
        return self._tone(text)

    def _tone(self, text: str) -> np.ndarray:
        seconds = min(8.0, 0.4 + 0.04 * len(text))
        t = np.arange(int(seconds * self.sample_rate)) / self.sample_rate
        return (0.2 * np.sin(2 * math.pi * 440 * t)).astype(np.float32)


class Narrator:
    """Chooses the engine per language, manages reference voices and joins chunks."""

    def __init__(
        self,
        primary: SpeechEngine,
        fallback_model_id: Optional[str],
        fallback_loader: Optional[Callable[[], SpeechEngine]],
        fallback_languages: Iterable[str],
        translator: Translator,
        voice_dir: Path,
        max_chunk_chars: int,
        reference_voice: Optional[Voice] = None,
    ) -> None:
        self.primary = primary
        self.fallback_model_id = fallback_model_id
        self._fallback_loader = fallback_loader
        self.fallback_languages: FrozenSet[str] = frozenset(fallback_languages)
        self._fallback: Optional[SpeechEngine] = None
        self.translator = translator
        self.voice_dir = voice_dir
        self.max_chunk_chars = max_chunk_chars
        self.reference_voice = reference_voice
        self._voices: Dict[Tuple[str, str], Voice] = {}
        self._lock = threading.Lock()
        voice_dir.mkdir(parents=True, exist_ok=True)

    @property
    def model_ids(self) -> List[str]:
        return [self.primary.model_id] + ([self.fallback_model_id] if self.fallback_model_id else [])

    def supports(self, language: str) -> bool:
        if self.primary.supports(language):
            return True
        return self._fallback_loader is not None and language in self.fallback_languages

    def narrate(self, text: str, language: str) -> Tuple[np.ndarray, int, str]:
        """Returns (mono float32 waveform, sample rate, model id)."""
        engine = self.engine_for(language)
        voice = self.voice_for(engine, language)
        chunks = chunk_text(text, self.max_chunk_chars)
        if not chunks:
            raise ValueError("Nothing to narrate: the text is empty.")

        pause = np.zeros(int(PAUSE_BETWEEN_CHUNKS_SECONDS * engine.sample_rate), dtype=np.float32)
        pieces = []
        for index, chunk in enumerate(chunks):
            if index:
                pieces.append(pause)
            pieces.append(engine.clone(chunk, language, voice))
        return np.concatenate(pieces), engine.sample_rate, engine.model_id

    def engine_for(self, language: str) -> SpeechEngine:
        if self.primary.supports(language):
            return self.primary
        if not self.supports(language):
            raise ValueError(f"No narration model is configured for '{language}'.")
        with self._lock:
            if self._fallback is None:
                logger.info("Loading fallback TTS model for '%s'", language)
                self._fallback = self._fallback_loader()
        return self._fallback

    def voice_for(self, engine: SpeechEngine, language: str) -> Voice:
        if self.reference_voice:
            return self.reference_voice

        key = (engine.model_id, language)
        if key in self._voices:
            return self._voices[key]

        stem = f"{engine.model_id.replace('/', '_')}-{language}"
        wav_path = self.voice_dir / f"{stem}.wav"
        meta_path = self.voice_dir / f"{stem}.json"
        if wav_path.exists() and meta_path.exists():
            voice = Voice(str(wav_path), json.loads(meta_path.read_text())["transcript"])
        else:
            transcript = self._sample_sentence(language)
            logger.info("Designing the %s reference voice with %s", language, engine.model_id)
            _write_wav(wav_path, engine.design(transcript, language), engine.sample_rate)
            meta_path.write_text(json.dumps({"transcript": transcript}, ensure_ascii=False))
            voice = Voice(str(wav_path), transcript)
        self._voices[key] = voice
        return voice

    def _sample_sentence(self, language: str) -> str:
        if language == "en":
            return VOICE_SAMPLE_SENTENCE
        try:
            return self.translator.translate(VOICE_SAMPLE_SENTENCE, "en", language)
        except Exception:  # noqa: BLE001 - an English sample still yields a usable voice
            logger.exception("Could not translate the voice sample into %s", language)
            return VOICE_SAMPLE_SENTENCE


def _write_wav(path: Path, wav: np.ndarray, sample_rate: int) -> None:
    import soundfile as sf

    sf.write(str(path), wav, sample_rate)

In [ ]:
%%writefile museum_ai/audio.py
"""Waveform -> compressed file for the webhook."""

from __future__ import annotations

import io
import shutil
import subprocess
import wave
from typing import Tuple

import numpy as np


def encode(wav: np.ndarray, sample_rate: int, bitrate: str = "64k") -> Tuple[bytes, str]:
    """Returns (file bytes, mime type): MP3 through ffmpeg, or WAV when ffmpeg is missing.

    Mono 64 kbps MP3 keeps a two-minute narration near 1 MB, small enough to
    travel base64-encoded inside the webhook JSON.
    """
    pcm = _to_pcm16(wav)
    if shutil.which("ffmpeg"):
        result = subprocess.run(
            [
                "ffmpeg", "-hide_banner", "-loglevel", "error",
                "-f", "s16le", "-ar", str(sample_rate), "-ac", "1", "-i", "pipe:0",
                "-codec:a", "libmp3lame", "-b:a", bitrate, "-f", "mp3", "pipe:1",
            ],
            input=pcm,
            capture_output=True,
            check=True,
        )  # fmt: skip
        return result.stdout, "audio/mpeg"

    buffer = io.BytesIO()
    with wave.open(buffer, "wb") as file:
        file.setnchannels(1)
        file.setsampwidth(2)
        file.setframerate(sample_rate)
        file.writeframes(pcm)
    return buffer.getvalue(), "audio/wav"


def duration_seconds(wav: np.ndarray, sample_rate: int) -> float:
    return round(len(wav) / sample_rate, 2)


def _to_pcm16(wav: np.ndarray) -> bytes:
    samples = np.asarray(wav, dtype=np.float32).reshape(-1)
    peak = float(np.max(np.abs(samples))) if samples.size else 0.0
    if peak > 0:
        # Normalise to -1 dBFS so every language plays at a similar volume.
        samples = samples * (0.89 / peak)
    return (np.clip(samples, -1.0, 1.0) * 32767).astype("<i2").tobytes()

In [ ]:
%%writefile museum_ai/backend_client.py
"""Signed calls to the backend API: heartbeats and result webhooks."""

from __future__ import annotations

import json
import logging
import threading
from collections import deque
from dataclasses import dataclass
from typing import Any, Callable, Deque, Dict, List, Optional

import httpx

from .signing import SIGNATURE_HEADER, sign

logger = logging.getLogger(__name__)

PING_PATH = "/ai-services/ping"
HEARTBEAT_PATH = "/ai-services/heartbeat"
WEBHOOK_PATH = "/ai-services/webhooks/localization"


class BackendClient:
    def __init__(self, api_url: str, secret: str, timeout: float = 60.0) -> None:
        self.api_url = api_url.rstrip("/")
        self.secret = secret
        self.http = httpx.Client(timeout=timeout)

    def post(self, path: str, payload: Dict[str, Any]) -> httpx.Response:
        # The signature covers these exact bytes, so serialise once and send them as-is.
        body = json.dumps(payload, ensure_ascii=False, separators=(",", ":")).encode()
        return self.http.post(
            f"{self.api_url}{path}",
            content=body,
            headers={"Content-Type": "application/json", SIGNATURE_HEADER: sign(self.secret, body)},
        )

    def ping(self) -> httpx.Response:
        return self.post(PING_PATH, {})

    def heartbeat(self, payload: Dict[str, Any]) -> httpx.Response:
        return self.post(HEARTBEAT_PATH, payload)

    def send_event(self, event: Dict[str, Any]) -> httpx.Response:
        return self.post(WEBHOOK_PATH, event)

    def close(self) -> None:
        self.http.close()


@dataclass
class _Pending:
    event: Dict[str, Any]
    attempts: int = 0


class Outbox:
    """Delivers final task results in order, retrying while the backend is unreachable.

    A result that the backend rejects (4xx) will never be accepted, so a rejected
    `task.completed` is replaced by `task.failed` carrying the reason - otherwise
    the backend would wait for the task forever and eventually re-dispatch it.
    """

    def __init__(self, client: BackendClient, max_backoff: float = 60.0) -> None:
        self.client = client
        self.max_backoff = max_backoff
        self._items: Deque[_Pending] = deque()
        self._condition = threading.Condition()
        self._stopped = False

    def put(self, event: Dict[str, Any]) -> None:
        with self._condition:
            self._items.append(_Pending(event))
            self._condition.notify()

    def task_ids(self) -> List[str]:
        with self._condition:
            return [item.event["taskId"] for item in self._items]

    def __len__(self) -> int:
        with self._condition:
            return len(self._items)

    def stop(self) -> None:
        with self._condition:
            self._stopped = True
            self._condition.notify_all()

    def run(self) -> None:
        while True:
            with self._condition:
                while not self._items and not self._stopped:
                    self._condition.wait()
                if self._stopped:
                    return
                item = self._items[0]

            outcome = self._deliver(item)
            with self._condition:
                if outcome == "retry":
                    delay = min(self.max_backoff, 2 ** min(item.attempts, 6))
                    self._condition.wait(timeout=delay)
                    continue
                if outcome == "replace":
                    continue
                self._items.popleft()

    def _deliver(self, item: _Pending) -> str:
        event = item.event
        item.attempts += 1
        try:
            response = self.client.send_event(event)
        except httpx.HTTPError as error:
            logger.warning("Webhook for %s failed (attempt %s): %s", event["taskId"], item.attempts, error)
            return "retry"

        if response.status_code < 300:
            logger.info("Reported %s for task %s", event["event"], event["taskId"])
            return "done"
        if response.status_code >= 500 or response.status_code in (408, 429):
            logger.warning("Backend answered %s for %s; retrying", response.status_code, event["taskId"])
            return "retry"

        reason = _error_message(response)
        logger.error("Backend rejected %s for %s: %s", event["event"], event["taskId"], reason)
        if event["event"] == "task.completed" and response.status_code != 404:
            item.event = failed_event(event["taskId"], f"The API rejected the result: {reason}")
            item.attempts = 0
            return "replace"
        return "done"


def failed_event(task_id: str, error: str) -> Dict[str, Any]:
    return {"event": "task.failed", "taskId": task_id, "error": error[:2000]}


def _error_message(response: httpx.Response) -> str:
    try:
        body = response.json()
        message = body.get("message")
        details = body.get("details")
        return f"{message} {details}" if details else str(message)
    except ValueError:
        return response.text[:300]


def heartbeat_loop(
    client: BackendClient,
    payload_factory: Callable[[], Dict[str, Any]],
    interval: float,
    stop: threading.Event,
) -> None:
    """Announces this service every `interval` seconds until `stop` is set."""
    was_ok: Optional[bool] = None
    while not stop.is_set():
        try:
            response = client.heartbeat(payload_factory())
            ok = response.status_code < 300
            if ok and was_ok is not True:
                logger.info("Connected to the backend at %s", client.api_url)
            if not ok:
                logger.warning("Heartbeat rejected (%s): %s", response.status_code, _error_message(response))
            was_ok = ok
        except httpx.HTTPError as error:
            if was_ok is not False:
                logger.warning("Backend unreachable at %s: %s", client.api_url, error)
            was_ok = False
        stop.wait(interval)

In [ ]:
%%writefile museum_ai/worker.py
"""The queue: one GPU worker thread processing tasks in arrival order."""

from __future__ import annotations

import base64
import logging
import queue
import threading
from dataclasses import dataclass
from typing import Any, Callable, Dict, List, Optional

from .audio import duration_seconds, encode
from .backend_client import Outbox, failed_event
from .schemas import JobRequest, TaskSpec
from .text import narration_text
from .translation import Translator, translate_copy
from .tts import Narrator

logger = logging.getLogger(__name__)

#: Sends a progress event; returns the backend's JSON answer, or None when unreachable.
ProgressSink = Callable[[Dict[str, Any]], Optional[Dict[str, Any]]]


class TaskCancelled(Exception):
    """The backend no longer wants this task (superseded, failed or deleted)."""


@dataclass
class _QueuedTask:
    job: JobRequest
    task: TaskSpec


class LocalizationWorker:
    """Translate -> narrate -> report, one task at a time.

    Progress events (`task.processing`) are best effort, and double as a
    cancellation check: when the backend answers that the task was superseded,
    the GPU work is skipped. Final events go through the outbox, which retries
    until the backend accepts them. Task ids stay "active" until their final
    event is delivered, so the backend does not re-dispatch work that is merely
    waiting for delivery.
    """

    def __init__(
        self,
        translator: Translator,
        narrator: Narrator,
        outbox: Outbox,
        send_progress: ProgressSink,
        gpu_lock: threading.Lock,
        mp3_bitrate: str = "64k",
    ) -> None:
        self.translator = translator
        self.narrator = narrator
        self.outbox = outbox
        self.send_progress = send_progress
        self.gpu_lock = gpu_lock
        self.mp3_bitrate = mp3_bitrate
        self._queue: "queue.Queue[Optional[_QueuedTask]]" = queue.Queue()
        self._active: Dict[str, str] = {}
        self._lock = threading.Lock()
        self._thread: Optional[threading.Thread] = None

    # ---- intake -------------------------------------------------------------

    def submit(self, job: JobRequest) -> List[str]:
        """Queues the job's tasks; ids already queued or awaiting delivery are ignored."""
        accepted: List[str] = []
        with self._lock:
            pending_delivery = set(self.outbox.task_ids())
            for task in job.tasks:
                if task.taskId in self._active or task.taskId in pending_delivery:
                    continue
                self._active[task.taskId] = "queued"
                self._queue.put(_QueuedTask(job, task))
                accepted.append(task.taskId)
        if accepted:
            languages = ", ".join(task.languageCode for task in job.tasks if task.taskId in accepted)
            logger.info("Queued %s [%s]", job.exhibitCode, languages)
        return accepted

    def active_task_ids(self) -> List[str]:
        with self._lock:
            active = list(self._active)
        return active + [task_id for task_id in self.outbox.task_ids() if task_id not in active]

    def queue_size(self) -> int:
        with self._lock:
            return len(self._active)

    # ---- processing ---------------------------------------------------------

    def start(self) -> None:
        self._thread = threading.Thread(target=self._run, name="localization-worker", daemon=True)
        self._thread.start()

    def stop(self) -> None:
        self._queue.put(None)

    def _run(self) -> None:
        while True:
            item = self._queue.get()
            if item is None:
                return
            event: Optional[Dict[str, Any]]
            try:
                event = self.process(item.job, item.task)
            except TaskCancelled as reason:
                logger.info("Skipped %s/%s: %s", item.job.exhibitCode, item.task.languageCode, reason)
                event = None
            except Exception as error:  # noqa: BLE001 - never let one task kill the queue
                logger.exception("Task %s crashed", item.task.taskId)
                event = failed_event(item.task.taskId, f"{type(error).__name__}: {error}")
            # Under the lock, so submit() always sees the id in one of the two places.
            with self._lock:
                if event is not None:
                    self.outbox.put(event)
                self._active.pop(item.task.taskId, None)

    def process(self, job: JobRequest, task: TaskSpec) -> Dict[str, Any]:
        language = task.languageCode
        logger.info("Processing %s/%s", job.exhibitCode, language)
        self._check_supported(job, task)
        with self._lock:
            self._active[task.taskId] = "processing"

        with self.gpu_lock:
            if task.translate:
                self._progress(task, "translating")
                copy = translate_copy(self.translator, job.source, language)
            else:
                copy = {
                    "title": job.source.title,
                    "shortDescription": job.source.shortDescription,
                    "description": job.source.description,
                }

            self._progress(task, "synthesizing")
            script = narration_text(copy["title"], copy["shortDescription"], copy["description"])
            wav, sample_rate, tts_model = self.narrator.narrate(script, language)

        audio, mime_type = encode(wav, sample_rate, self.mp3_bitrate)
        logger.info(
            "Finished %s/%s: %.1fs of audio (%d KB)",
            job.exhibitCode,
            language,
            duration_seconds(wav, sample_rate),
            len(audio) // 1024,
        )
        event: Dict[str, Any] = {
            "event": "task.completed",
            "taskId": task.taskId,
            "audio": {
                "base64": base64.b64encode(audio).decode(),
                "mimeType": mime_type,
                "durationSeconds": duration_seconds(wav, sample_rate),
            },
            "models": {
                "translation": self.translator.model_id if task.translate else None,
                "tts": tts_model,
            },
        }
        if task.translate:
            event["translation"] = {key: value for key, value in copy.items() if value is not None}
        return event

    def _check_supported(self, job: JobRequest, task: TaskSpec) -> None:
        """Fails fast, before any GPU work, with a message staff can act on."""
        if task.translate:
            for code in (job.source.languageCode, task.languageCode):
                if code not in self.translator.languages:
                    raise ValueError(
                        f"The translation model {self.translator.model_id} does not support '{code}'."
                    )
        if not self.narrator.supports(task.languageCode):
            raise ValueError(f"No narration model on this AI service supports '{task.languageCode}'.")

    def _progress(self, task: TaskSpec, stage: str) -> None:
        try:
            answer = self.send_progress({"event": "task.processing", "taskId": task.taskId, "stage": stage})
        except TaskCancelled:
            raise
        except Exception as error:  # noqa: BLE001 - progress is informational only
            logger.warning("Could not report %s for %s: %s", stage, task.taskId, error)
            return
        if answer is not None and answer.get("applied") is False:
            raise TaskCancelled(answer.get("reason") or "no longer requested")

In [ ]:
%%writefile museum_ai/server.py
"""HTTP intake: the backend POSTs jobs here."""

from __future__ import annotations

import json
from typing import Callable, Dict

from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse
from pydantic import ValidationError

from .schemas import JobRequest
from .signing import SIGNATURE_HEADER, verify
from .worker import LocalizationWorker


def create_app(secret: str, worker: LocalizationWorker, describe: Callable[[], Dict]) -> FastAPI:
    app = FastAPI(title="Smart Museum Guide AI service", docs_url=None, redoc_url=None)

    @app.get("/health")
    def health() -> Dict:
        return describe()

    @app.post("/v1/jobs")
    async def submit_job(request: Request) -> JSONResponse:
        body = await request.body()
        if not verify(secret, request.headers.get(SIGNATURE_HEADER), body):
            return JSONResponse({"message": "Invalid or missing signature."}, status_code=401)
        try:
            job = JobRequest.model_validate_json(body)
        except ValidationError as error:
            details = json.loads(error.json(include_url=False))
            return JSONResponse({"message": "Invalid job.", "details": details}, status_code=400)

        accepted = worker.submit(job)
        return JSONResponse({"accepted": accepted, "queueSize": worker.queue_size()}, status_code=202)

    return app

In [ ]:
%%writefile museum_ai/tunnel.py
"""Cloudflare quick tunnel: a public https URL for a Colab runtime, no account needed."""

from __future__ import annotations

import logging
import os
import platform
import re
import stat
import subprocess
import threading
import time
import urllib.request
from pathlib import Path
from typing import Tuple

logger = logging.getLogger(__name__)

_URL = re.compile(r"https://[a-z0-9-]+\.trycloudflare\.com")
_RELEASES = "https://github.com/cloudflare/cloudflared/releases/latest/download"


def start_quick_tunnel(port: int, bin_dir: Path, timeout: float = 90.0) -> Tuple[subprocess.Popen, str]:
    """Starts `cloudflared tunnel --url http://127.0.0.1:<port>` and returns (process, public URL)."""
    binary = _cloudflared(bin_dir)
    process = subprocess.Popen(
        [binary, "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{port}"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    found: list = []
    ready = threading.Event()

    def pump() -> None:
        # Keep draining the output for the tunnel's whole life, or cloudflared blocks.
        for line in process.stdout:  # type: ignore[union-attr]
            match = _URL.search(line)
            if match and not found:
                found.append(match.group(0))
                ready.set()
        ready.set()

    threading.Thread(target=pump, name="cloudflared-output", daemon=True).start()
    if not ready.wait(timeout) or not found:
        process.terminate()
        raise RuntimeError("cloudflared did not report a public URL. Set PUBLIC_URL to use another tunnel.")

    # The hostname needs a moment to resolve before the first request arrives.
    time.sleep(3)
    return process, found[0]


def _cloudflared(bin_dir: Path) -> str:
    for directory in os.environ.get("PATH", "").split(os.pathsep):
        candidate = Path(directory) / "cloudflared"
        if candidate.exists():
            return str(candidate)

    machine = platform.machine().lower()
    asset = "cloudflared-linux-arm64" if machine in ("aarch64", "arm64") else "cloudflared-linux-amd64"
    target = bin_dir / "cloudflared"
    if not target.exists():
        bin_dir.mkdir(parents=True, exist_ok=True)
        logger.info("Downloading %s", asset)
        urllib.request.urlretrieve(f"{_RELEASES}/{asset}", target)
        target.chmod(target.stat().st_mode | stat.S_IEXEC)
    return str(target)

In [ ]:
%%writefile museum_ai/service.py
"""Wires everything together: models, queue, HTTP server, tunnel and heartbeats."""

from __future__ import annotations

import logging
import platform
import subprocess
import threading
import time
import uuid
from pathlib import Path
from typing import Any, Dict, List, Optional

import httpx
import uvicorn

from . import __version__
from .audio import encode
from .backend_client import BackendClient, Outbox, heartbeat_loop
from .config import Settings
from .languages import supported_languages
from .server import create_app
from .translation import load_translator
from .tts import MockEngine, Narrator, OmniVoiceEngine, SpeechEngine, Voice, VoxCPM2Engine
from .tunnel import start_quick_tunnel
from .worker import LocalizationWorker, TaskCancelled

logger = logging.getLogger("museum_ai")


class _ThreadedServer(uvicorn.Server):
    """uvicorn inside a background thread (the notebook keeps its main thread)."""

    def install_signal_handlers(self) -> None:  # pragma: no cover - older uvicorn only
        pass


class AiService:
    def __init__(self, settings: Settings) -> None:
        self.settings = settings
        self.instance_id = uuid.uuid4().hex
        self.work_dir = Path(settings.work_dir).resolve()
        self.gpu_lock = threading.Lock()
        self.public_url: Optional[str] = None
        self.device = _device_name()
        self._stop = threading.Event()
        self._tunnel: Optional[subprocess.Popen] = None
        self._server: Optional[_ThreadedServer] = None

        self.client = BackendClient(settings.backend_api_url, settings.secret)
        self.outbox = Outbox(self.client)
        self.translator = None
        self.narrator: Optional[Narrator] = None
        self.worker: Optional[LocalizationWorker] = None

    # ---- lifecycle ------------------------------------------------------------

    def start(self) -> "AiService":
        self.work_dir.mkdir(parents=True, exist_ok=True)
        _configure_logging(self.work_dir / "service.log")
        self._check_backend()
        self._load_models()

        self.worker = LocalizationWorker(
            translator=self.translator,
            narrator=self.narrator,
            outbox=self.outbox,
            send_progress=self._send_progress,
            gpu_lock=self.gpu_lock,
            mp3_bitrate=self.settings.mp3_bitrate,
        )
        self.worker.start()
        threading.Thread(target=self.outbox.run, name="webhook-outbox", daemon=True).start()

        self._start_http()
        if self.settings.public_url:
            self.public_url = self.settings.public_url
        else:
            self._tunnel, self.public_url = start_quick_tunnel(self.settings.port, self.work_dir / "bin")
        logger.info("AI service reachable at %s", self.public_url)

        threading.Thread(
            target=heartbeat_loop,
            args=(self.client, self._heartbeat_payload, self.settings.heartbeat_seconds, self._stop),
            name="heartbeat",
            daemon=True,
        ).start()
        return self

    def stop(self) -> None:
        self._stop.set()
        if self.worker:
            self.worker.stop()
        self.outbox.stop()
        if self._server:
            self._server.should_exit = True
        if self._tunnel:
            self._tunnel.terminate()
        logger.info("AI service stopped")

    def languages(self) -> List[str]:
        """Languages both loaded models cover; the CMS offers exactly these."""
        if not self.translator or not self.narrator:
            return []
        return supported_languages(self.translator.languages, self.narrator.supports)

    def status(self) -> Dict[str, Any]:
        return {
            "instanceId": self.instance_id,
            "version": __version__,
            "publicUrl": self.public_url,
            "backendApiUrl": self.settings.backend_api_url,
            "device": self.device,
            "translationModel": getattr(self.translator, "model_id", None),
            "ttsModels": self.narrator.model_ids if self.narrator else [],
            "languages": self.languages(),
            "queueSize": self.worker.queue_size() if self.worker else 0,
            "activeTaskIds": self.worker.active_task_ids() if self.worker else [],
            "pendingWebhooks": len(self.outbox),
        }

    # ---- listening test -------------------------------------------------------

    def preview(
        self, text: str, language: str, source_language: Optional[str] = None, filename: Optional[str] = None
    ) -> Path:
        """Translates (when `source_language` differs) and narrates `text` without the backend.

        Returns the path of an audio file to play in the notebook.
        """
        assert self.narrator and self.translator, "Call start() first."
        with self.gpu_lock:
            if source_language and source_language != language:
                text = self.translator.translate(text, source_language, language)
                logger.info("Translation (%s): %s", language, text)
            wav, sample_rate, _ = self.narrator.narrate(text, language)
        audio, mime_type = encode(wav, sample_rate, self.settings.mp3_bitrate)
        extension = ".mp3" if mime_type == "audio/mpeg" else ".wav"
        path = self.work_dir / "previews" / (filename or f"preview-{language}-{int(time.time())}{extension}")
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(audio)
        return path

    # ---- internals ----------------------------------------------------------------

    def _check_backend(self) -> None:
        try:
            response = self.client.ping()
        except httpx.HTTPError as error:
            logger.warning(
                "Cannot reach BACKEND_API_URL=%s (%s). The service will keep retrying.",
                self.settings.backend_api_url,
                error,
            )
            return
        if response.status_code == 401:
            raise RuntimeError(
                "The backend rejected the signature: AI_SERVICE_SECRET differs from backend/api/.env."
            )
        if response.status_code == 503:
            raise RuntimeError(
                "AI_SERVICE_SECRET is not set in backend/api/.env - set it and restart the API."
            )
        if response.status_code == 404:
            raise RuntimeError(
                f"{self.settings.backend_api_url}/ai-services/ping was not found. "
                "BACKEND_API_URL must end with /api."
            )
        if response.status_code < 300:
            logger.info("Backend API reachable at %s", self.settings.backend_api_url)

    def _load_models(self) -> None:
        settings = self.settings
        start = time.time()
        self.translator = load_translator(
            settings.translation_model, settings.translation_quantization, settings.mock_models
        )

        primary: SpeechEngine
        fallback_loader = None
        fallback_id = None
        if settings.mock_models:
            primary = MockEngine()
        else:
            logger.info("Loading TTS model %s", settings.tts_model)
            primary = VoxCPM2Engine(
                settings.tts_model,
                settings.voice_description,
                settings.narration_style,
                settings.voxcpm_optimize,
            )
            if settings.fallback_tts_model:
                fallback_id = settings.fallback_tts_model
                fallback_loader = lambda: OmniVoiceEngine(fallback_id, settings.omnivoice_instruct)  # noqa: E731

        reference_voice = None
        if settings.reference_voice_path:
            if not settings.reference_voice_text:
                raise ValueError(
                    "REFERENCE_VOICE_TEXT (the transcript) is required with REFERENCE_VOICE_PATH."
                )
            reference_voice = Voice(settings.reference_voice_path, settings.reference_voice_text)

        self.narrator = Narrator(
            primary=primary,
            fallback_model_id=fallback_id,
            fallback_loader=fallback_loader,
            fallback_languages=settings.fallback_tts_languages,
            translator=self.translator,
            voice_dir=self.work_dir / "voices",
            max_chunk_chars=settings.max_chunk_chars,
            reference_voice=reference_voice,
        )
        logger.info("Models ready in %.0fs on %s", time.time() - start, self.device)
        logger.info("Offering %d languages: %s", len(self.languages()), ", ".join(self.languages()))

    def _start_http(self) -> None:
        app = create_app(self.settings.secret, self.worker, self.status)
        config = uvicorn.Config(app, host=self.settings.host, port=self.settings.port, log_level="warning")
        self._server = _ThreadedServer(config)
        threading.Thread(target=self._server.run, name="http", daemon=True).start()
        deadline = time.time() + 30
        while not self._server.started:
            if time.time() > deadline:
                raise RuntimeError(f"The HTTP server did not start on port {self.settings.port}.")
            time.sleep(0.1)

    def _heartbeat_payload(self) -> Dict[str, Any]:
        return {
            "instanceId": self.instance_id,
            "url": self.public_url,
            "queueSize": self.worker.queue_size() if self.worker else 0,
            "activeTaskIds": self.worker.active_task_ids() if self.worker else [],
            "translationModel": getattr(self.translator, "model_id", None) or self.settings.translation_model,
            "ttsModels": self.narrator.model_ids if self.narrator else [self.settings.tts_model],
            "languages": self.languages(),
            "device": self.device,
        }

    def _send_progress(self, event: Dict[str, Any]) -> Optional[Dict[str, Any]]:
        try:
            response = self.client.send_event(event)
        except httpx.HTTPError as error:
            logger.warning("Progress update failed: %s", error)
            return None
        if response.status_code == 404:
            raise TaskCancelled("the task no longer exists")
        if response.status_code >= 300:
            logger.warning("Progress update rejected (%s): %s", response.status_code, response.text[:200])
            return None
        return response.json()


def _device_name() -> str:
    try:
        import torch

        if torch.cuda.is_available():
            properties = torch.cuda.get_device_properties(0)
            return f"{properties.name} ({properties.total_memory / 1024**3:.0f} GB)"
    except ImportError:
        pass
    return f"CPU ({platform.machine()})"


def _configure_logging(log_file: Path) -> None:
    """Logs to the console and to a file: Colab shows background-thread output in whichever cell runs."""
    root = logging.getLogger("museum_ai")
    if root.handlers:
        return
    formatter = logging.Formatter("%(asctime)s %(levelname)s %(name)s: %(message)s", "%H:%M:%S")
    for handler in (logging.StreamHandler(), logging.FileHandler(log_file, encoding="utf-8")):
        handler.setFormatter(formatter)
        root.addHandler(handler)
    root.setLevel(logging.INFO)
    root.propagate = False


def main() -> None:
    service = AiService(Settings.from_env()).start()
    try:
        while True:
            time.sleep(3600)
    except KeyboardInterrupt:
        service.stop()


if __name__ == "__main__":
    main()

## Run

In [ ]:
#@title Start the service
import sys

try:
    service.stop()  # re-running this cell replaces the previous instance
except NameError:
    pass
for name in [module for module in list(sys.modules) if module.startswith("museum_ai")]:
    del sys.modules[name]  # pick up edited %%writefile cells

from museum_ai.config import Settings
from museum_ai.service import AiService

service = AiService(Settings.from_env()).start()
service.status()

In [ ]:
#@title Listening test (does not touch the CMS)
TEXT = "Trống đồng Đông Sơn là biểu tượng tiêu biểu của nền văn minh Việt cổ."  #@param {type:"string"}
SOURCE_LANGUAGE = "vi"  #@param {type:"string"}
TARGET_LANGUAGE = "en"  #@param {type:"string"}

from IPython.display import Audio, display

path = service.preview(TEXT, TARGET_LANGUAGE, source_language=SOURCE_LANGUAGE)
display(Audio(str(path)))

In [ ]:
#@title Keep running and show the log (stop this cell to use the notebook)
import json, time

log_file = f"{os.environ['WORK_DIR']}/service.log"
position = 0
while True:
    with open(log_file, encoding="utf-8") as file:
        file.seek(position)
        chunk = file.read()
        position = file.tell()
    if chunk:
        print(chunk, end="")
    time.sleep(10)

### Troubleshooting

| Symptom | Fix |
| --- | --- |
| `The backend rejected the signature` | `AI_SERVICE_SECRET` here and in `backend/api/.env` differ. |
| `AI_SERVICE_SECRET is not set in backend/api/.env` | Set it, restart the API. |
| `Cannot reach BACKEND_API_URL` | The API tunnel is down or the URL changed; restart `cloudflared` and update the form. |
| CMS shows *AI service offline* | This notebook stopped, or the tunnel died. Run all again; waiting tasks resume. |
| `401 ... gated repo` while loading TranslateGemma | Accept the license on Hugging Face and add `HF_TOKEN`. |
| CUDA out of memory | Use an L4/A100 runtime, keep `translategemma-4b-it`, or clear `FALLBACK_TTS_MODEL`. |
| A language sounds wrong | Try the listening test; change `VOICE_DESCRIPTION`, then delete `/content/museum-ai/voices` so reference voices are recreated, and press *Regenerate* in the CMS. |